# Pipeline para Generación de Agonistas de PD-L1

Este notebook implementa un pipeline completo para generar agonistas de PD-L1 usando:
- **RF Diffusion**: Generación de estructuras proteicas
- **Protein MPNN**: Diseño de secuencias
- **AlphaFold3**: Validación de estructuras

## 1. Configuración e Importaciones

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Encuentra el root del proyecto buscando una carpeta "src" hacia arriba
p = Path.cwd().resolve()
project_root = None

for _ in range(10):  # sube hasta 10 niveles
    if (p / "src").is_dir():
        project_root = p
        break
    p = p.parent

if project_root is None:
    raise FileNotFoundError("No encontré la carpeta 'src' en ningún padre del directorio actual.")

sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("Added to sys.path:", sys.path[0])

from src.rf_diffusion_generator import RFDiffusionGenerator, generate_multiple_structures
from src.protein_mpnn_generator import ProteinMPNNGenerator, generate_sequences_for_multiple_structures
from src.alphafold3_predictor import AlphaFold3Predictor, validate_mpnn_sequences_with_alphafold3

Project root: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/pd-l1-drug-accelerator
Added to sys.path: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/pd-l1-drug-accelerator


In [2]:
# Configuración de rutas
data_dir = project_root / "data"
raw_data_dir = data_dir / "raw"
processed_data_dir = data_dir / "processed"

# Crear directorios necesarios
processed_data_dir.mkdir(parents=True, exist_ok=True)

# Rutas específicas para PD-L1
pdl1_data_dir = processed_data_dir / "pdl1_agonists"
pdl1_data_dir.mkdir(parents=True, exist_ok=True)

## 4. Paso 3: Validación con AlphaFold3

Validamos que las secuencias generadas por Protein MPNN se plieguen correctamente usando AlphaFold3.

In [3]:
from pathlib import Path
from typing import Tuple, List, Optional
import pandas as pd


def read_fasta_to_df(
    fasta_path: str | Path,
    id_col: str = "sequence_id",
    seq_col: str = "sequence",
    keep_full_header: bool = False,
) -> pd.DataFrame:
    """
    Lee un archivo FASTA y regresa un DataFrame con columnas:
    - sequence_id
    - sequence

    Args:
        fasta_path: ruta al archivo .fasta/.fa
        id_col: nombre de la columna para el ID
        seq_col: nombre de la columna para la secuencia
        keep_full_header: si True, usa todo el header sin truncar; si False, usa el primer token

    Returns:
        pd.DataFrame
    """
    fasta_path = Path(fasta_path)
    if not fasta_path.exists():
        raise FileNotFoundError(f"No existe el archivo FASTA: {fasta_path}")

    records: List[Tuple[str, str]] = []
    current_id: Optional[str] = None
    current_seq_parts: List[str] = []

    def flush_record():
        nonlocal current_id, current_seq_parts
        if current_id is None:
            return
        seq = "".join(current_seq_parts).replace(" ", "").replace("\t", "").upper()
        if seq:
            records.append((current_id, seq))
        current_id = None
        current_seq_parts = []

    with fasta_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                flush_record()
                header = line[1:].strip()
                current_id = header if keep_full_header else header.split()[0]
            else:
                current_seq_parts.append(line)

    flush_record()

    df = pd.DataFrame(records, columns=[id_col, seq_col])

    # Limpieza mínima / validación
    df = df.drop_duplicates(subset=[id_col]).reset_index(drop=True)

    # Si no hay IDs o vienen vacíos, generamos IDs automáticos
    if df.empty:
        return df

    if df[id_col].isna().any() or (df[id_col].astype(str).str.len() == 0).any():
        df[id_col] = [f"seq_{i:04d}" for i in range(len(df))]

    return df


In [4]:

# -------------------------
# USO: poblar all_sequences_df desde FASTA
# -------------------------
fasta_file = "../../data/raw/candidates.fa"  # <-- ajusta a tu ruta

all_sequences_df = read_fasta_to_df(
    fasta_path=fasta_file,
    id_col="sequence_id",
    seq_col="sequence",
    keep_full_header=False
)

print(f"FASTA cargado: {len(all_sequences_df)} secuencias")
display(all_sequences_df.head())

FASTA cargado: 2 secuencias


,sequence_id,sequence
0,"rfdiffusion_1769486473,",AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...
1,"T=0.1,",AFKVTAPKTEYVVELGSDVSLSCNFPVEGKLDLSKLTVVWTKHGEL...


In [5]:
from pathlib import Path

# AJUSTA SOLO ESTO a donde clonaste alphafold3
# /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/
#AF3_REPO_ROOT = Path.home() / "mnt" / "e" / "Documentos" / "Maestria" / "Proyecto Integrador" / "Codigo" /"alphafold3"   # <-- cambia si tu ruta es otra
AF3_REPO_ROOT = Path("/mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/alphafold3")

# Script de ejecución (según el snippet que pusiste)
alphafold3_path = AF3_REPO_ROOT / "run_alphafold.py"

# Params (esto depende de cómo bajaste los weights/params; pongo 2 opciones comunes)
# Opción A: params dentro del repo
model_params_dir = AF3_REPO_ROOT / "params"
# Opción B: params en una carpeta aparte (si así los descargaste)
# model_params_dir = Path.home() / "alphafold3_params"

# Checks claros para no fallar a medio pipeline
if not alphafold3_path.exists():
    raise FileNotFoundError(f"No encontré run_alphafold.py en: {alphafold3_path}")
if not model_params_dir.exists():
    raise FileNotFoundError(f"No encontré el directorio de params en: {model_params_dir}")

print("AlphaFold3 script:", alphafold3_path)
print("Model params dir:", model_params_dir)


AlphaFold3 script: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/alphafold3/run_alphafold.py
Model params dir: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/alphafold3/params


In [6]:
import shutil, os
print("PATH:", os.environ.get("PATH"))
print("jackhmmer:", shutil.which("jackhmmer"))
print("hhsearch:", shutil.which("hhsearch"))


PATH: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/venv/bin:/root/.vscode-server/bin/b6a47e94e326b5c209d118cf0f994d6065585705/bin/remote-cli:/mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/venv/bin:/root/miniforge3/bin:/root/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/usr/lib/wsl/lib:/mnt/c/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.4/bin:/mnt/c/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.4/libnvvp:/mnt/c/Python312/Scripts:/mnt/c/Python312:/mnt/c/WINDOWS/system32:/mnt/c/WINDOWS:/mnt/c/WINDOWS/System32/Wbem:/mnt/c/WINDOWS/System32/WindowsPowerShell/v1.0:/mnt/c/WINDOWS/System32/OpenSSH:/mnt/c/Program Files/Git/cmd:/mnt/c/Program Files/PuTTY:/mnt/c/ProgramData/chocolatey/bin:/mnt/c/Users/Nitro/AppData/Roaming/nvm:/mnt/c/Program Files/nodejs:/mnt/c/Program Files/Java/jdk-17/bin:/mnt/c/Program Files (x86)/NVIDIA Corporation/PhysX/Common:/mnt/c/Program Files/NVIDIA Corporation/NVIDIA app/NvDLISR

In [7]:
all_sequences_df = all_sequences_df.drop(all_sequences_df.index[1]).reset_index(drop=True)

print(all_sequences_df.columns)

# Caso 1: si tu FASTA se cargó con "id"
if "sequence_id" not in all_sequences_df.columns and "id" in all_sequences_df.columns:
    all_sequences_df = all_sequences_df.rename(columns={"id": "sequence_id"})

# Caso 2: si tu FASTA se cargó con "header" o "name"
elif "sequence_id" not in all_sequences_df.columns and "header" in all_sequences_df.columns:
    all_sequences_df = all_sequences_df.rename(columns={"header": "sequence_id"})

elif "sequence_id" not in all_sequences_df.columns and "name" in all_sequences_df.columns:
    all_sequences_df = all_sequences_df.rename(columns={"name": "sequence_id"})


Index(['sequence_id', 'sequence'], dtype='str')


In [8]:
import sys
import subprocess
import json
from pathlib import Path
from typing import Optional, List, Dict
import pandas as pd


# Encuentra el root del proyecto buscando una carpeta "src" hacia arriba
p = Path.cwd().resolve()
project_root = None

for _ in range(10):  # sube hasta 10 niveles
    if (p / "src").is_dir():
        project_root = p
        break
    p = p.parent

if project_root is None:
    raise FileNotFoundError("No encontré la carpeta 'src' en ningún padre del directorio actual.")

sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("Added to sys.path:", sys.path[0])

from src.rf_diffusion_generator import RFDiffusionGenerator, generate_multiple_structures
from src.protein_mpnn_generator import ProteinMPNNGenerator, generate_sequences_for_multiple_structures
from src.alphafold3_predictor import AlphaFold3Predictor, validate_mpnn_sequences_with_alphafold3

af3_predictor = AlphaFold3Predictor(
    alphafold3_path=alphafold3_path,
    model_params_dir=model_params_dir,
    db_dir="/data/alphafold"  # <-- AJUSTA si tu carpeta se llama distinto
)



# Validar secuencias (ahora vienen del FASTA)
if 'all_sequences_df' in locals() and not all_sequences_df.empty:
    print("Validando secuencias con AlphaFold3...")

    validation_df = validate_mpnn_sequences_with_alphafold3(
        mpnn_sequences_df=all_sequences_df,
        predictor=af3_predictor,
        output_dir=Path("data/processed/alphafold3_validation")
    )

    print(f"\nValidación completada para {len(validation_df)} secuencias")
    display(validation_df.head())

    # Filtrar secuencias con predicción exitosa
    successful_predictions = validation_df[validation_df["status"] == "success"]
    print(f"\nSecuencias con predicción exitosa: {len(successful_predictions)}")
else:
    print("No hay secuencias para validar")


Project root: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/pd-l1-drug-accelerator
Added to sys.path: /mnt/e/Documentos/Maestria/Proyecto Integrador/Codigo/pd-l1-drug-accelerator
Validando secuencias con AlphaFold3...
Validando 1 secuencias con AlphaFold3...
Prediciendo estructura para rfdiffusion_1769486473,...
mpnn_sequences_df cols: ['sequence_id', 'sequence']
validation_df cols: ['sequence_id', 'status', 'returncode', 'json_file', 'stdout_log', 'stderr_log', 'output_dir', 'pdb_file', 'mmcif_file']
validation_df index name: None
validation_df head:
               sequence_id   status  returncode  \
0  rfdiffusion_1769486473  success           0   

                                           json_file  \
0  /mnt/e/Documentos/Maestria/Proyecto Integrador...   

                                          stdout_log  \
0  /mnt/e/Documentos/Maestria/Proyecto Integrador...   

                                          stderr_log  \
0  /mnt/e/Documentos/Maestria/Proyecto Integrador.

,sequence_id,sequence,status,returncode,json_file,stdout_log,stderr_log,output_dir,pdb_file,mmcif_file
0,"rfdiffusion_1769486473,",AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Secuencias con predicción exitosa: 0


In [9]:
# Cargar utilidades del proyecto existente
from src.ifeature_process import calcular_descriptores_ifeature
from src.PeptideBert_predict import predict_peptidebert

# Filtrar candidatos exitosos
if 'successful_predictions' in locals() and not successful_predictions.empty:
    print(f"Analizando {len(successful_predictions)} candidatos...")
    
    # Calcular descriptores moleculares
    print("\nCalculando descriptores moleculares...")
    
    # Preparar DataFrame para iFeature
    sequences_df = successful_predictions[['sequence_id', 'sequence']].copy()
    sequences_df.columns = ['ID', 'sequence']
    
    # Calcular descriptores (esto requiere archivos FASTA)
    # Primero guardar como FASTA
    from src.bio_utils import save_df_as_fasta
    
    fasta_file = pdl1_data_dir / "candidate_sequences.fasta"
    save_df_as_fasta(
        dataframe=sequences_df,
        id_col='ID',
        seq_col='sequence',
        output_file=fasta_file
    )
    
    # Calcular descriptores
    descriptors_df = calcular_descriptores_ifeature(
        directorio_temporal=pdl1_data_dir / "temp",
        dataframe=sequences_df,
        sequence_col='sequence',
        id_col='ID'
    )
    
    # Evaluar propiedades farmacológicas con PeptideBERT
    print("\nEvaluando propiedades farmacológicas...")
    
    # Nota: Ajusta la ruta al modelo PeptideBERT según tu instalación
    peptidebert_path = project_root / "models" / "peptideBert"
    
    if peptidebert_path.exists():
        properties_df = predict_peptidebert(
            model_directory_path=str(peptidebert_path),
            input_dataframe=sequences_df,
            sequence_col='sequence',
            feats=['hemo', 'sol', 'nf']  # Hemólisis, solubilidad, no adherencia
        )
        
        # Combinar todos los resultados
        final_df = successful_predictions.merge(
            descriptors_df, left_on='sequence_id', right_on='ID', how='left'
        ).merge(
            properties_df, left_on='sequence_id', right_on='ID', how='left'
        )
        
        # Guardar resultados finales
        output_file = pdl1_data_dir / "pdl1_agonist_candidates.csv"
        final_df.to_csv(output_file, index=False)
        
        print(f"\nResultados guardados en: {output_file}")
        print(f"Total de candidatos: {len(final_df)}")
        
        # Mostrar mejores candidatos (ejemplo: alta solubilidad, baja hemólisis)
        if 'sol' in final_df.columns and 'hemo' in final_df.columns:
            best_candidates = final_df[
                (final_df['sol'] > 0.7) &  # Alta solubilidad
                (final_df['hemo'] < 0.1)   # Baja hemólisis
            ].sort_values('sol', ascending=False)
            
            print(f"\nMejores candidatos (alta solubilidad, baja hemólisis): {len(best_candidates)}")
            print(best_candidates[['sequence_id', 'sequence', 'sol', 'hemo']].head(10))
    else:
        print(f"Modelo PeptideBERT no encontrado en {peptidebert_path}")
else:
    print("No hay candidatos exitosos para analizar")

ModuleNotFoundError: No module named 'iFeatureOmega'

## 6. Próximos Pasos

1. **Docking Molecular**: Usar herramientas como AutoDock Vina para predecir afinidad con PD-L1
2. **Análisis de Estabilidad**: Evaluar estabilidad térmica y enzimática
3. **Síntesis y Validación Experimental**: Seleccionar los mejores candidatos para síntesis